# Binary Quantisation (1-bit)

Purpose: Implements binary quantisation manually using `sign(w)` in the forward pass with a straight-through estimator (STE) in the backward pass. The same STE pattern used for QAT, but mapping weights to {-1, +1} rather than int8 levels.

Approach:
1. Rebuild the baseline architecture using custom `BinaryConv2D` / `BinaryDense` layers that binarise weights (`sign(w)` -> {-1, +1}) in the forward pass, with STE gradients (clipped to the [-1, 1] range, standard BNN convention) in the backward pass.
2. Fine-tune from the FP32 baseline's pretrained weights.
3. Export via the standard TFLite converter without int8 optimisation. The model stays float32, but all conv/dense kernel values are now exactly -1 or +1.
4. Evaluate using the same pattern as PTQ/QAT (adjusted for float32 input/output, since there's no int8 quantisation step here).

In [ ]:
# import
import tensorflow as tf
import numpy as np
import os
import time
import csv

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/tinyml-quant-security'

In [ ]:
# Load CIFAR-10 and reapply the same fixed split as in other notebooks
(x_train_full, y_train_full), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
x_train_full = x_train_full.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
y_train_full = y_train_full.flatten()
y_test = y_test.flatten()

split = np.load(f'{PROJECT_DIR}/results/data_split.npz')
train_idx, val_idx = split['train_idx'], split['val_idx']

x_train, y_train = x_train_full[train_idx], y_train_full[train_idx]
x_val, y_val = x_train_full[val_idx], y_train_full[val_idx]

print(f"Train: {x_train.shape}, Val: {x_val.shape}, Test: {x_test.shape}")

In [ ]:
# Same tf.data augmentation pipeline as baseline_training notebook and QAT notebook
BATCH_SIZE = 64

def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.resize_with_crop_or_pad(image, 36, 36)
    image = tf.image.random_crop(image, size=[32, 32, 3])
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = train_ds.shuffle(len(x_train), seed=SEED).map(augment, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

## Binarisation with straight-through estimator (STE)

- Forward pass: `sign(w)` maps every weight to exactly -1 or +1.
- Backward pass: gradient passes through unchanged, but only where `|w| <= 1` which is the standard BinaryConnect/BNN convention, may it different from the unclipped STE used for QAT's int8 rounding

**Note** Without clipping, weights could be very large values (e.g. +50 or −50) during training. The sign of +50 and +0.1 are both +1

In [ ]:
@tf.custom_gradient
def binarize_ste(w):
    w_binarized = tf.sign(w)
    # tf.sign(0) == 0, which is not a valid binary value, map it to +1
    w_binarized = tf.where(tf.equal(w_binarized, 0), tf.ones_like(w_binarized), w_binarized)

    def grad(dy):
        # Straight-through estimator, clipped: gradient passes through only
        # where |w| <= 1, zeroed elsewhere (standard BNN convention)
        mask = tf.cast(tf.abs(w) <= 1.0, dy.dtype)
        return dy * mask

    return w_binarized, grad

In [ ]:
class BinaryConv2D(tf.keras.layers.Conv2D):
    """Conv2D with binarised weights (forward: sign, backward: clipped STE)."""
    def call(self, inputs):
        q_kernel = binarize_ste(self.kernel)
        outputs = tf.nn.conv2d(
            inputs, q_kernel, strides=[1, *self.strides, 1],
            padding=self.padding.upper()
        )
        if self.use_bias:
            outputs = tf.nn.bias_add(outputs, self.bias)
        if self.activation is not None:
            outputs = self.activation(outputs)
        return outputs


class BinaryDense(tf.keras.layers.Dense):
    """Dense with binarised weights (forward: sign, backward: clipped STE)."""
    def call(self, inputs):
        q_kernel = binarize_ste(self.kernel)
        outputs = tf.matmul(inputs, q_kernel)
        if self.use_bias:
            outputs = tf.nn.bias_add(outputs, self.bias)
        if self.activation is not None:
            outputs = self.activation(outputs)
        return outputs

**Note:** Unlike the QAT notebook, activations here are left in full precision because of standard ReLU. Binarising activations is also a larger accuracy hit and a bigger implementation step (sign-based activations need a different backward pass too). So, my decision is to evaluate 'binary weights' only, which is still a meaningful and common extreme quantisation configuration

In [ ]:
# Binary model architecture
# First and last layers kept in full precision
def build_binary_model(num_classes=10):
    inputs = tf.keras.layers.Input(shape=(32, 32, 3))

    x = tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu', name='conv1')(inputs)
    x = tf.keras.layers.BatchNormalization(name='bn1')(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)

    x = BinaryConv2D(64, 3, padding='same', activation='relu', name='conv2')(x)
    x = tf.keras.layers.BatchNormalization(name='bn2')(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)

    x = BinaryConv2D(128, 3, padding='same', activation='relu', name='conv3')(x)
    x = tf.keras.layers.BatchNormalization(name='bn3')(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)

    x = tf.keras.layers.Flatten()(x)
    x = BinaryDense(128, activation='relu', name='dense1')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax', name='dense2')(x)

    return tf.keras.Model(inputs, outputs)

binary_model = build_binary_model()
binary_model.summary()

## Load pretrained baseline weights and fine-tune

In [ ]:
baseline_model = tf.keras.models.load_model(f'{PROJECT_DIR}/models/baseline.keras')

baseline_weight_layers = [l for l in baseline_model.layers if l.weights]
binary_weight_layers = [l for l in binary_model.layers if l.weights]

assert len(baseline_weight_layers) == len(binary_weight_layers), (
    f"Layer count mismatch: baseline has {len(baseline_weight_layers)} weighted layers, "
    f"binary model has {len(binary_weight_layers)}. Check both architectures match."
)

for src, dst in zip(baseline_weight_layers, binary_weight_layers):
    dst.set_weights(src.get_weights())

print("Transferred baseline weights into binary model.")

In [ ]:
binary_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

pre_finetune_loss, pre_finetune_acc = binary_model.evaluate(x_test, y_test)
print(f"Binary model BEFORE fine-tuning - Test accuracy: {pre_finetune_acc:.4f}")
print("(Expect a larger drop here than QAT saw - binarisation is a much harsher")
print(" perturbation than int8 rounding, so the untrained binary weights should")
print(" hurt accuracy noticeably before fine-tuning adapts to it.)")

In [ ]:
# Fine-tune with binarisation active
BINARY_EPOCHS = 25

binary_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=3, min_lr=1e-6),
]

binary_history = binary_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=BINARY_EPOCHS,
    callbacks=binary_callbacks
)

In [ ]:
post_finetune_loss, post_finetune_acc = binary_model.evaluate(x_test, y_test)
print(f"Binary model AFTER fine-tuning - Test accuracy: {post_finetune_acc:.4f}")

# Confirm weights are actually binary now
sample_kernel = binary_model.get_layer('conv2').kernel.numpy()
unique_signs = np.unique(np.sign(sample_kernel))
print(f"Unique sign values in conv2 kernel (underlying float weights, before binarize_ste forward pass): {unique_signs}")
print("Note: stored weights remain float32 - binarize_ste() is applied at inference/training time in call(),")
print("not stored as a separate binarised copy. This is what Approach B means by 'float32 storage'.")

In [ ]:
binary_model.save(f'{PROJECT_DIR}/models/binary_model.keras')

## Export for deployment

Rebuild an architecture where the binary layers' weights are baked in as their binarised (+-1) values, for example, we apply `sign()` once, store the result as a normal Conv2D/Dense kernel, and export that.
This way the exported model truly has +-1 weight values (not the underlying pre-binarisation float weights), while still using standard Conv2D/Dense ops the TFLite converter can trace

In [ ]:
def build_plain_model(num_classes=10):
    inputs = tf.keras.layers.Input(shape=(32, 32, 3))
    x = tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu', name='conv1')(inputs)
    x = tf.keras.layers.BatchNormalization(name='bn1')(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)

    x = tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu', name='conv2')(x)
    x = tf.keras.layers.BatchNormalization(name='bn2')(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)

    x = tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu', name='conv3')(x)
    x = tf.keras.layers.BatchNormalization(name='bn3')(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)

    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(128, activation='relu', name='dense1')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax', name='dense2')(x)
    return tf.keras.Model(inputs, outputs)

plain_binary_model = build_plain_model()

binary_layer_names = {'conv2', 'conv3', 'dense1'}  # the layers that were Binary* in binary_model

for src_layer in binary_model.layers:
    if not src_layer.weights:
        continue
    dst_layer = plain_binary_model.get_layer(src_layer.name)
    weights = src_layer.get_weights()

    if src_layer.name in binary_layer_names:
        # Apply sign() once to bake in the actual +-1 values (kernel is weights[0])
        kernel = weights[0]
        kernel_binarized = np.sign(kernel)
        kernel_binarized[kernel_binarized == 0] = 1.0
        weights[0] = kernel_binarized.astype(np.float32)

    dst_layer.set_weights(weights)

print("Built plain model with baked-in +-1 weights for conv2, conv3, dense1.")

In [ ]:
# Recheck the accuracy
plain_binary_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
_, plain_check_acc = plain_binary_model.evaluate(x_test, y_test)
print(f"Plain (baked-in +-1 weights) binary model -- Test accuracy: {plain_check_acc:.4f}")
print(f"Should closely match fine-tuned binary model accuracy: {post_finetune_acc:.4f}")

plain_binary_model.export(f'{PROJECT_DIR}/models/binary_savedmodel')

In [ ]:
# Export to TFLite without int8 optimisation
converter = tf.lite.TFLiteConverter.from_saved_model(f'{PROJECT_DIR}/models/binary_savedmodel')
# Deliberately not setting converter.optimizations where this keeps the model float32
# Weight values are still exactly +-1 for the binarised layers
binary_tflite_model = converter.convert()

binary_model_path = f'{PROJECT_DIR}/models/binary_model.tflite'
with open(binary_model_path, 'wb') as f:
    f.write(binary_tflite_model)

print(f"Saved binary TFLite model to {binary_model_path}")
print(f"Binary model size: {os.path.getsize(binary_model_path) / 1024:.2f} KB")

## Clean evaluation (float32 input/output - no int8 quantisation step needed here)

In [ ]:
interpreter = tf.lite.Interpreter(model_path=binary_model_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]
print(f"Input dtype: {input_details['dtype']}")


def evaluate_tflite_float_model(interpreter, x_test, y_test, num_samples=None):
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    if num_samples is None:
        num_samples = len(x_test)

    correct = 0
    latencies = []

    for i in range(num_samples):
        x_sample = x_test[i:i+1].astype(np.float32)

        start = time.perf_counter()
        interpreter.set_tensor(input_details['index'], x_sample)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details['index'])
        latencies.append(time.perf_counter() - start)

        pred = np.argmax(output[0])
        if pred == y_test[i]:
            correct += 1

    accuracy = correct / num_samples
    avg_latency_ms = np.mean(latencies) * 1000
    p95_latency_ms = np.percentile(latencies, 95) * 1000

    return accuracy, avg_latency_ms, p95_latency_ms


binary_accuracy, binary_avg_latency, binary_p95_latency = evaluate_tflite_float_model(
    interpreter, x_test, y_test
)
binary_size_kb = os.path.getsize(binary_model_path) / 1024

print(f"Binary - Accuracy: {binary_accuracy:.4f}")
print(f"Binary - Avg latency: {binary_avg_latency:.3f} ms, P95 latency: {binary_p95_latency:.3f} ms")
print(f"Binary - Model size: {binary_size_kb:.2f} KB")

In [ ]:
# Append to results CSV
results_path = f'{PROJECT_DIR}/results/clean_eval.csv'
file_exists = os.path.isfile(results_path)

with open(results_path, 'a', newline='') as f:
    writer = csv.writer(f)
    if not file_exists:
        writer.writerow(['variant', 'accuracy', 'avg_latency_ms', 'p95_latency_ms', 'size_kb'])
    writer.writerow(['Binary', binary_accuracy, binary_avg_latency, binary_p95_latency, binary_size_kb])

print(f"Appended Binary results to {results_path}")